In [1]:
# packages
import pandas as pd
import numpy as np
import matplotlib as mpl
#import chartgazing as cg
from bs4 import BeautifulSoup
import requests
import json
import re
import time

In [11]:
# pull all set names
url = "https://www.pokellector.com/sets"

res = requests.get(url)
soup = BeautifulSoup(res.content, "html.parser")

packs = soup.select('span')

card_sets = []

for i, pack in enumerate(packs):
    try:
        card_sets.append(pack.get_text())
    except Exception as e:
        print(f"Error: {e}")

card_sets

card_sets = card_sets[1:-1]

card_sets

['Delta Reign',
 '30th Celebration (English)',
 '30th Celebration Classic Collection',
 '30th Celebration Pikachu Collection',
 'Pitch Black',
 'Chaos Rising',
 'Perfect Order',
 'Ascended Heroes',
 'Phantasmal Flames',
 'Mega Evolution',
 'Mega Evolution Energies',
 'Mega Evolution Black Star Promos',
 'White Flare',
 'Black Bolt',
 'Destined Rivals',
 'Journey Together',
 "McDonald's Dragon Discovery",
 'Prismatic Evolutions',
 'Surging Sparks',
 'Stellar Crown',
 'Holiday Calendar 2024',
 'Trick or Trade 2024',
 'Shrouded Fable',
 'Twilight Masquerade',
 'Temporal Forces',
 'Paldean Fates',
 'Paradox Rift',
 'Scarlet & Violet - 151',
 'Holiday Calendar 2023',
 'Trick or Trade 2023',
 "McDonald's Match Battle 2023",
 'Obsidian Flames',
 'Paldea Evolved',
 'Scarlet & Violet Energies',
 'Scarlet & Violet',
 'Scarlet & Violet Promos',
 'Crown Zenith',
 'Crown Zenith - Galarian Gallery',
 'Silver Tempest - Trainer Gallery',
 'Silver Tempest',
 'Lost Origin',
 'Lost Origin - Trainer Galle

In [ ]:
# add a set name column for then pulling different sets from pokecollector


# import pokemon names and numbers
url = "https://www.pokellector.com/Base-Set-Expansion/"

res = requests.get(url)
soup = BeautifulSoup(res.content, "html.parser")

pokemon = soup.select('.plaque')

pokemon_names = []

for entry in pokemon:
    try:
        pokemon_names.append(entry.get_text())
    except Exception as e:
        print(f"An error occurred: {e}")
        
pokemon_df = pd.DataFrame(pokemon_names, columns=['pokemon'])

pokemon_df["number"] = pokemon_df["pokemon"].str.extract(r"#\s*([^\s-]+)", expand=False)
pokemon_df["name"] = pokemon_df["pokemon"].str.extract(r"\-(.*)", expand=False).str.strip().str.lower()

pokemon_df.loc[pokemon_df["name"] == "nidoran m", "name"] = "nidoran"

pokemon_df

['#1 - Alakazam', '#2 - Blastoise', '#3 - Chansey', '#4 - Charizard', '#5 - Clefairy']


,pokemon,number,name
0,#1 - Alakazam,1,alakazam
1,#2 - Blastoise,2,blastoise
2,#3 - Chansey,3,chansey
3,#4 - Charizard,4,charizard
4,#5 - Clefairy,5,clefairy
...,...,...,...
97,#98 - Fire Energy,98,fire energy
98,#99 - Grass Energy,99,grass energy
99,#100 - Lightning Energy,100,lightning energy
100,#101 - Psychic Energy,101,psychic energy


In [ ]:
def poke_pull(name, number):
    """Pull a Pokemon cards value history from pricecharting.com"""
    url = f"https://www.pricecharting.com/game/pokemon-base-set/{name}-1st-edition-{number}"

    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text

    chart_data = json.loads(re.search(r"VGPC\.chart_data\s*=\s*(\{.*?\});", html, re.S).group(1))

    df = pd.DataFrame(
        [(grade, ts, price) for grade, points in chart_data.items() for ts, price, in points],
        columns =["grade", "date", "price"]
    )

    df["date"] = pd.to_datetime(df["date"], unit="ms")
    df["price"] = df["price"] / 100
    df["name"] = name
    df["number"] = number

    return df

print(poke_pull(pokemon_df["name"][0], pokemon_df["number"][0]))

       grade                date   price      name number
0    boxonly 2020-12-01 07:00:00    0.00  alakazam      1
1    boxonly 2021-01-01 07:00:00    0.00  alakazam      1
2    boxonly 2021-02-01 07:00:00    0.00  alakazam      1
3    boxonly 2021-03-01 07:00:00    0.00  alakazam      1
4    boxonly 2021-04-01 06:00:00    0.00  alakazam      1
..       ...                 ...     ...       ...    ...
421     used 2026-06-01 06:00:00  550.00  alakazam      1
422     used 2026-07-01 06:00:00  677.18  alakazam      1
423     used 2026-08-01 06:00:00  648.00  alakazam      1
424     used 2026-09-01 06:00:00  597.32  alakazam      1
425     used 2026-10-01 06:00:00  770.00  alakazam      1

[426 rows x 5 columns]


In [ ]:
frames = []

for name, number in zip(pokemon_df["name"], pokemon_df["number"]):
    try:
        frames.append(poke_pull(name, number))
        print(f"Successfull added #{number} - {name}")

        time.sleep(5)
    except Exception as e:
        print(f"Error occured: {e}")

pokemon_all = pd.concat(frames, ignore_index=True)

pokemon_all["number"] = pokemon_all["number"].astype(int)

Successfull added #1 - alakazam
Successfull added #2 - blastoise
Successfull added #3 - chansey
Successfull added #4 - charizard
Successfull added #5 - clefairy
Successfull added #6 - gyarados
Successfull added #7 - hitmonchan
Successfull added #8 - machamp
Successfull added #9 - magneton
Successfull added #10 - mewtwo
Successfull added #11 - nidoking
Successfull added #12 - ninetales
Successfull added #13 - poliwrath
Successfull added #14 - raichu
Successfull added #15 - venusaur
Successfull added #16 - zapdos
Successfull added #17 - beedrill
Successfull added #18 - dragonair
Successfull added #19 - dugtrio
Successfull added #20 - electabuzz
Successfull added #21 - electrode
Successfull added #22 - pidgeotto
Successfull added #23 - arcanine
Successfull added #24 - charmeleon
Successfull added #25 - dewgong
Successfull added #26 - dratini
Successfull added #27 - farfetch'd
Successfull added #28 - growlithe
Successfull added #29 - haunter
Successfull added #30 - ivysaur
Successfull adde

In [74]:
print(pokemon_all.dtypes)
print(pokemon_all.info())

pokemon_all

grade                str
date      datetime64[ms]
price            float64
name                 str
number             int64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 42534 entries, 0 to 42533
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   grade   42534 non-null  str           
 1   date    42534 non-null  datetime64[ms]
 2   price   42534 non-null  float64       
 3   name    42534 non-null  str           
 4   number  42534 non-null  int64         
dtypes: datetime64[ms](1), float64(1), int64(1), str(2)
memory usage: 1.6 MB
None


,grade,date,price,name,number
0,boxonly,2020-12-01 07:00:00,0.00,alakazam,1
1,boxonly,2021-01-01 07:00:00,0.00,alakazam,1
2,boxonly,2021-02-01 07:00:00,0.00,alakazam,1
3,boxonly,2021-03-01 07:00:00,0.00,alakazam,1
4,boxonly,2021-04-01 06:00:00,0.00,alakazam,1
...,...,...,...,...,...
42529,used,2026-06-01 06:00:00,13.74,water energy,102
42530,used,2026-07-01 06:00:00,11.50,water energy,102
42531,used,2026-08-01 06:00:00,13.19,water energy,102
42532,used,2026-09-01 06:00:00,13.67,water energy,102
